# sqrt-eps-stabilize — worked example 3: One RMSprop step: accumulate variance, apply stabilized update

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

RMSprop maintains a running exponential moving average of squared gradients (v ← β*v + (1-β)*g²). The parameter update is `p -= lr * g / sqrt(v + eps)`. The eps is placed inside the sqrt for the same reason as Adam: it bounds the effective learning rate for near-zero second moments, preventing explosive updates when a parameter has seen nearly no gradient signal.

## Worked solution

**Step 1 — Initialize parameter, second moment, and hyperparameters.** We use a 4-element parameter vector so we can see different second moment values.

**Step 2 — Simulate one forward pass: assign gradients.** In real training these come from `.backward()`; here we set them manually.

**Step 3 — Update second moment.** `v = beta * v + (1 - beta) * g**2`.

**Step 4 — Compute the stabilized step.** `step = lr * g / sqrt(v + eps)`. Apply in-place: `p -= step`.

**Step 5 — Print the effective learning rates.** `lr / sqrt(v + eps)` shows how the adaptive scaling works — large accumulated gradient → small effective lr.

In [ ]:
import torch as t

t.manual_seed(7)

# RMSprop parameters
beta = 0.99
lr   = 0.01
eps  = 1e-8

p = t.tensor([1.0, 2.0, -0.5, 3.0])  # parameters
v = t.zeros_like(p)                   # second moment estimates (start at 0)

# Simulated gradients for this step
g = t.tensor([0.5, 2.0, 0.1, 4.0])

# RMSprop second moment update
v = beta * v + (1.0 - beta) * g ** 2

# Stabilized parameter update
step = lr * g / t.sqrt(v + eps)
p = p - step

eff_lr = lr / t.sqrt(v + eps)
print('Second moments v:', v.tolist())
print('Effective LR    :', [f'{x:.4f}' for x in eff_lr.tolist()])
print('Updated params  :', p.tolist())
print('All finite:', t.isfinite(p).all().item())